<h2 style="color: #0f2027; background: linear-gradient(90deg, #43cea2 0%, #185a9d 100%); padding: 12px 0; border-radius: 8px; text-align:center; font-size: 2rem; letter-spacing: 1px;">
   <span style="color: #fff;">Introduction to Ray Data</span> 
</h2>

This notebook will provide an overview of Ray Data and how to use it to read, transform and write data in a distributed manner.

<div class="alert alert-block alert-info">
<b> Here is the roadmap for this notebook:</b>
<ul>
  <li>When and why to use Ray Data?</li>
  <li>How to work with Ray Data</li>
  <li>Loading data</li>
  <li>Lazy evaluation mode</li>
  <li>Transforming data</li>
  <li>Stateful transformations with Ray Actors</li>
  <li>Materializing data</li>
  <li>Persisting data</li>
</ul>
</div>

**Imports**

In [ ]:
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import ray
import torch
from torchvision.transforms import Compose, Normalize, ToTensor

## 1. When to Consider Ray Data

Use Ray Data to load and preprocess data for distributed ML workloads. Compared to other loading solutions, Datasets are more flexible and provide [higher overall performance](https://www.anyscale.com/blog/why-third-generation-ml-platforms-are-more-performant). Ray Data is especially performant when needing to run pre-processing in a **streaming fashion** across a **large dataset** on a **heterogeneous cluster of CPUs and GPUs**.


Use Datasets as a last-mile bridge from storage or ETL pipeline outputs to distributed applications and libraries in Ray. 

<img src='https://docs.ray.io/en/releases-2.34.0/_images/dataset-loading-1.svg' width=60%/>

Consider Ray Data for your project if it fits at least one of the following scenarios:

<table>
  <thead>
    <tr>
      <th>Scenario</th>
      <th>Challenge</th>
      <th>How Ray Data Helps</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td rowspan="2"><strong>Large-scale data or model processing</strong></td>
      <td>You need to load or process very large datasets (e.g., over 10 TB).</td>
      <td>Distributes data loading and computation across a Ray cluster.</td>
    </tr>
    <tr>
      <td>You want to run inference with large models, such as LLMs.</td>
      <td>Integrates with inference engines for large models via <a href="https://docs.ray.io/en/latest/data/working-with-llms.html">ray.data.llm</a>.</td>
    </tr>
    <tr>
      <td rowspan="3"><strong>Maximizing CPU and GPU utilization</strong></td>
      <td>You currently over‑provision resources just to split or partition the data.</td>
      <td>Streams data through the pipeline to avoid materializing the entire dataset into cluster memory.</td>
    </tr>
    <tr>
      <td>You use fixed resource allocations at the start of the pipeline.</td>
      <td>Dynamically shares CPUs and GPUs across pipeline stages.</td>
    </tr>
    <tr>
      <td>Your pipeline stages run separately on CPU and GPU, writing intermediate results to disk.</td>
      <td>Enables pipeline parallelism with configurable batch sizes and avoids disk I/O bottlenecks.</td>
    </tr>
    <tr>
      <td rowspan="2"><strong>Building fault-tolerant pipelines</strong></td>
      <td>You must handle transient failures like network issues, spot instance interruptions, or hardware errors.</td>
      <td>Uses Ray Core’s built-in fault tolerance to retry failed tasks.</td>
    </tr>
    <tr>
      <td>You need to resume pipelines from the last checkpoint after failures.</td>
      <td>Offers driver checkpointing (with RayTurbo) to restart from the last saved state.</td>
    </tr>
    <tr>
      <td rowspan="2"><strong>Efficiently processing unstructured data (e.g. text, images, audio, video)</strong></td>
      <td>Your data is skewed, requiring repartition that demands a large amount of cluster memory.</td>
      <td>Automatically splits data into uniform blocks to balance work across the cluster.</td>
    </tr>
    <tr>
      <td>You need automatic scaling of resources to handle fluctuating workloads.</td>
      <td>Supports autoscaling for both CPU and GPU resources based on demand.</td>
    </tr>
  </tbody>
</table>



Here is an example batch inference pipeline with Ray Data over a very large image dataset.

<img src="https://docs.ray.io/en/releases-2.6.1/_images/stream-example.png" width="800" loading="lazy">

Note how:
1. The pipeline is split into two multiple stages which run on different hardware (CPU and GPU)
2. The data is streamed through the pipeline to avoid materializing the entire dataset into cluster memory.

## 2. How to work with Ray Data

It is commonly a three step process when using Ray Data:
1. Create your Dataset (most commonly using an IO connector)
2. Apply transformations to your Dataset
3. Consume your Dataset by either:
   1. Writing it out to a sink (file-based or database)
   2. Iterating over it (connecting it to a training process)

## 3. Loading data

Datasets uses Ray tasks to read data from remote storage. When reading from a file-based datasource (e.g., S3, GCS), it creates a number of read tasks proportional to the number of CPUs in the cluster. Each read task reads its assigned files and produces an output block:

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-summit/rag-app/dataset-read-cropped-v2.svg" width="500px">

Our dataset is a collection of hand-written digits (`MNIST`) located in cloud blob storage (on AWS S3).

In [ ]:
# Here is our dataset it contains 50 images per class
!aws s3 ls s3://anyscale-public-materials/ray-ai-libraries/mnist/50_per_index/

Let's create a Ray Dataset using the `read_images` function

In [ ]:
ds = ray.data.read_images("s3://anyscale-public-materials/ray-ai-libraries/mnist/50_per_index/", include_paths=True)
type(ds)

<div class="alert alert-block alert-info">
  <p><strong>Ray Data supports a variety of data sources for loading data</strong></p>
  <ul>
    <li>
      Reading files from common file formats (e.g. Parquet, CSV, JSON, etc.)
      <ul>
          <li><code>ds = ray.data.read_parquet("s3://...")</code></li>
      </ul>
    </li>
    <li>Loading from in-memory data structures (e.g. NumPy, PyTorch, etc.)
      <ul>
          <li><code>ray.data.from_torch(torch_ds)</code></li>
      </ul>
    <li>Loading from data lakehouses and warehouses such as Snowflake, Iceberg, and Databricks.</li>
      <ul>
          <li><code>ds = ray.data.read_databricks_tables()</code></li>
      </ul>
  </ul>
  <p>
    Start with an extensive list of <a href="https://docs.ray.io/en/latest/data/api/input_output.html#input-output" target="_blank">supported formats</a> and review further options in our <a href="https://docs.ray.io/en/latest/data/loading-data.html#loading-data" target="_blank">data loading guide</a>.
  </p>
</div>

Under the hood, Ray Data will use Ray tasks to read data from remote storage.

|<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-data-deep-dive/Ray+Data+Internals+-+reading.png" width="500px" loading="lazy">|
|:--|
|When reading from a file-based datasource, Ray Data starts with a number of read tasks proportional to the number of CPUs in the cluster. |
|Each read task reads its assigned files and produces output blocks.|

### 3.1 Note on blocks

|<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-intro/block.png" width="700px" loading="lazy">|
|:--|
|A Dataset when materialized is a distributed collection of blocks. This example illustrates a materialized dataset with three blocks, each block holding 1000 rows.|

<div class="alert alert-block alert-info">
A <strong>block</strong> is a contiguous subset of rows from a dataset. Blocks are distributed across the cluster and processed independently in parallel. By default blocks are PyArrow tables.
</div>

## 4. Lazy evaluation mode

In Ray Data, operations are not executed immediately. Most transformations are **lazy**, meaning they build up an execution plan rather than running right away. 

The execution plan is only **executed** when you call a method that *materializes* or *consumes* the dataset.

To trigger a limited execution to only materialize a small subset of the data, you can use the `take_batch` method.

In [ ]:
batch = ds.take_batch(batch_size=3)
batch

Let's visualize an example image:

In [ ]:
img = batch["image"][0]
title = batch["path"][0]

plt.title(title)
plt.axis("off")
plt.imshow(img, cmap='gray')

<div class="alert alert-block alert-info">

<b>Note on execution triggering methods in Ray Dataset</b>

To determinte if an operation will trigger execution, look for the methods with the `ConsumptionAPI` decorator in the [`Dataset.py`](https://github.com/ray-project/ray/blob/master/python/ray/data/dataset.py).

These categories of operations trigger execution (with some examples):
* Method designed to consume Datasets for **writing**:
  * [`write_parquet`](https://docs.ray.io/en/latest/data/api/doc/ray.data.Dataset.write_parquet.html#ray.data.Dataset.write_parquet)
* Method designed to consume Datasets for **distributed training**:
  * [`streaming_split`](https://github.com/ray-project/ray/blob/master/python/ray/data/dataset.py#L1694)
* Methods that attempt to **show** data, for example:
  * [`take`](https://docs.ray.io/en/latest/data/api/doc/ray.data.Dataset.take.html#ray.data.Dataset.take)
  * [`show`](https://docs.ray.io/en/latest/data/api/doc/ray.data.Dataset.show.html#ray-data-dataset-show)
* **Aggregations**, which attempt to reduce a dataset to a single value per column:
  * [`min`](https://docs.ray.io/en/latest/data/api/doc/ray.data.Dataset.min.html#ray.data.Dataset.min)
  * [`sum`](https://docs.ray.io/en/latest/data/api/doc/ray.data.Dataset.sum.html#ray.data.Dataset.sum)

Another way to trigger execution is to explicitly call <a href="https://docs.ray.io/en/latest/data/api/doc/ray.data.Dataset.materialize.html#ray-data-dataset-materialize" target="_blank">materialize()</a>. This will execute the underlying plan and generate the entire data blocks onto the cluster's memory.

## 5. Transforming data

To transform data, we can use the [`map_batches`](https://docs.ray.io/en/latest/data/api/doc/ray.data.Dataset.map_batches.html#ray-data-dataset-map-batches) API. 

`ds.map_batches` takes a function which accepts a batch of data and returns a batch of transformed data.

In [ ]:
def normalize(batch: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    transform = Compose([ToTensor(), Normalize((0.5,), (0.5,))])
    batch["image"] = [transform(image) for image in batch["image"]]
    return batch

<div class="alert alert-block alert-info">

**Note:** The default batch type is numpy-based i.e. `batch: dict[str, np.ndarray]`. 

You can also use:
- pandas-based batches by setting `batch_type="pandas"` i.e. `batch: pd.DataFrame`.
- pyarrow-based batches by setting `batch_type="pyarrow"` i.e. `batch: pa.Table`.

</div>

Calling `ds.map_batches` will add a `map_batches` operator to the execution plan.

In [ ]:
ds_normalized = ds.map_batches(normalize)

To tune the batch size for the transformation, specify the `batch_size` parameter.  

In [ ]:
ds_normalized = ds.map_batches(normalize, batch_size=32)

<div class="alert alert-block alert-info">

**Note:** batching only helps with performance when a transformation is vectorized - i.e. benefits from processing multiple rows at once.

Finding the optimal batch size depends on the hardware available and the target utilization.

</div>

### 5.1 On resource specification

To specify the exact resources to use for a `map_batches` transformation, specify the `num_cpus`, `num_gpus`, `memory`, and `resources` parameters.

- `num_cpus`: Number of CPUs to use for each task (use >1 if task performs multithreaded operations).
- `num_gpus`: Number of GPUs to use for each task.
- `memory`: Amount of RAM to use for each task (in bytes).
- `resources`: What is referred to as custom resources in Ray. It is a way to specify which node types to use for each task.

Let's specify 1 CPU and up to 100 MB of memory for each MapBatches task.

In [ ]:
ds_normalized = ds.map_batches(normalize, batch_size=32, num_cpus=1, memory=100 * 1024**2)

<div class="alert alert-block alert-info">

**Note:** Ray only performs a logical allocation of resources and does not physically enforce resource limits.

By default, Ray will [retry OOM errors](https://docs.ray.io/en/latest/ray-core/scheduling/ray-oom-prevention.html#retry-policy) and Ray Data will infinitely retry tasks that fail due to system failures.

Specifying resources helps avoid resource contention, avoiding unnecessary retries and confusing OOM errors.

</div>

To verify the output of `normalize()`, call [`take_batch()`](https://docs.ray.io/en/latest/data/api/doc/ray.data.Dataset.take_batch.html#ray.data.Dataset.take_batch) on the dataset.

In [ ]:
normalized_batch = ds_normalized.take_batch(batch_size=10)

Check the normalized pixel value range:

In [ ]:
for image in normalized_batch["image"]:
    assert image.shape == (1, 28, 28) # channel, height, width
    assert image.min() >= -1 and image.max() <= 1 # normalized to [-1, 1]

### 5.2 Activity: Implement a custom transformation

<div class="alert alert-block alert-info">

Here is what you need to do:

1. Add the ground truth label extracted from the image path.
    1. Note: the image path is in the format:`s3://anyscale-public-materials/ray-ai-libraries/mnist/50_per_index/{label}/{image_id}.png`.
2. Inspect the output of the transformation.


<details>
<summary>Click to view hints</summary>

Starting point:

```python
ds = ray.data.read_images("s3://anyscale-public-materials/ray-ai-libraries/mnist/50_per_index/", include_paths=True)
ds_normalized = ds.map_batches(normalize)

# Hint: Implement the add_label function
def add_label(batch: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    ...
    return batch

# Hint: Use `map_batches` to apply the add_label function
ds_labeled = ds_normalized.map_batches(add_label)

# Hint: Take a batch of the labeled dataset
labeled_batch = ds_labeled.take_batch(10)
print(labeled_batch)
```

</details>

</div>

In [ ]:
# Write your solution here

<div class="alert alert-block alert-info">

<details>

<summary>Click to view solution</summary>

```python
ds = ray.data.read_images("s3://anyscale-public-materials/ray-ai-libraries/mnist/50_per_index/", include_paths=True)
ds_normalized = ds.map_batches(normalize)

def add_label(batch: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    labels = []
    for item in batch["path"]:
        label = int(item.split("/")[-2])
        labels.append(label)

    batch["label"] = np.array(labels)
    return batch

ds_labeled = ds_normalized.map_batches(add_label)
labeled_batch = ds_labeled.take_batch(10)
print(labeled_batch)
```

</details>  
</div>

### 5.3 Autoscaling of stateless transformations

For stateless transformations, Ray Data will automatically scale up the number of tasks to match the number of input blocks within the available resource budget.

## 6. Stateful transformations with Ray Actors

In cases like batch inference, you want to spin up a number of actor processes that are **initialized once** with your model **and reused** to process multiple batches.

To implement this, you can use the `map_batches` API with a "Callable" class method that implements:

- `__init__`: Initialize any expensive state.
- `__call__`: Perform the stateful transformation.

For example, let's implement a `MNISTClassifier` that:
- loads a pre-trained model from a local file
- accepts a batch of images and generates the predicted label

In [ ]:
class MNISTClassifier:
    def __init__(self, remote_path: str, local_path: str):
        subprocess.run(f"aws s3 cp {remote_path} {local_path}", shell=True, check=True)

        self.model = torch.jit.load(local_path).to("cuda").eval()

    def __call__(self, batch: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
        images = torch.tensor(batch["image"]).float().to("cuda")

        with torch.no_grad():
            logits = self.model(images).cpu().numpy()

        batch["predicted_label"] = np.argmax(logits, axis=1)
        return batch

We can now use the `map_batches` API to apply the transformation to each batch of data.

Note, we specify 1 GPU per actor task and a total of 3 actors/workers.

In [ ]:
mnist_classifier_args = {
    "remote_path": "s3://anyscale-public-materials/ray-ai-libraries/mnist/model/model.pt",
    "local_path": "/mnt/cluster_storage/model.pt",
}

ds_preds = ds_normalized.map_batches(
    MNISTClassifier,
    fn_constructor_kwargs=mnist_classifier_args,
    num_gpus=1,
    concurrency=3,
    batch_size=100,
)

### 6.1 Resource specification for stateful transformations

It is common when you have varying hardware types in your cluster to want to further specify which accelerators to use for each stage of your pipeline.

Let's show how to achieve this with the `resources` parameter.

In [ ]:
ds_preds = ds_normalized.map_batches(
    MNISTClassifier,
    fn_constructor_kwargs=mnist_classifier_args,
    num_gpus=1,
    concurrency=3,
    batch_size=100,
    resources={"accelerator_type:T4": 0.0001},
)

<div class="alert alert-block alert-info">

<b>Note:</b> Pass in the Callable class uninitialized. Your driver will not execute the class constructor. Ray will pass in the arguments to the class constructor when the class is actually used in a transformation.


</div>

### 6.2 Autoscaling for stateful transformations

For stateful transformations, Ray Data will schedule tasks proportional to the number of actors (workers) in the pool. 

To specify an autoscaling pool, use a tuple of `(min_size, max_size)` for the `concurrency` parameter.

Ray Data will start with `min_size` actors and automatically scale up to `max_size` as needed.





In [ ]:
ds_preds = ds_normalized.map_batches(
    MNISTClassifier,
    fn_constructor_kwargs=mnist_classifier_args,
    num_gpus=1,
    concurrency=(1, 4),  # Autoscale pool based on blocks, resources and limits
    batch_size=100,
    resources={"accelerator_type:T4": 0.0001},
)

In [ ]:
batch_preds = ds_preds.take_batch(100)
batch_preds

## 7. Materializing data

You can choose to materialize the entire dataset into the Ray object store which is distributed across the cluster, primarily in memory and secondarily spilling to disk.

To materialize the dataset, we can use the `materialize()` method.

Use this **only** when you require the full dataset to compute downstream outputs.

In [ ]:
ds_preds = ds_preds.materialize()

`materialize()` triggers the execution. The logs should show the execution plan of Dataset:

```
Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> TaskPoolMapOperator[MapBatches(normalize)] -> ActorPoolMapOperator[MapBatches(MNISTClassifier)]
```

## 8. Persisting data

Finally, you can persist a dataset to storage using any of the "write" functions that Ray Data supports.

Lets write our predictions to a parquet dataset.

In [ ]:
ds_preds.write_parquet("/mnt/cluster_storage/mnist_preds")

Refer to the [Input/Output docs](https://docs.ray.io/en/latest/data/api/input_output.html) for a comprehensive list of write functions.